# ATLAS tutorial: prepare the wind station climatology

This notebook reads wind station observations, cleans the time series, computes a monthly climatology, joins the results with station metadata and saves the final table as a CSV file. The table is the input of the IDW notebook (4.1b).

It is designed as a step by step notebook for users who are not Python experts.

Normal users should only edit **Step 2**. All other cells can be run from top to bottom without changes.


## Step 1. Import required packages

Run this cell first.

If a package is missing, install it from a terminal or from a notebook cell. For example:

```bash
pip install pandas numpy matplotlib openpyxl geopandas
```

`geopandas` is only needed if you want to read and plot the shapefile.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


## Step 2. Input parameters

Edit only this section for normal use.

### Folder structure

The notebook reads and writes in `data/stations/<country>/`:

```text
data/stations/<country>/
├── stations_metadata.xlsx          station list (one row per station)
├── wind_speed_stations.csv         wind speed time series (m/s)
├── wind_direction_stations.csv     wind direction time series (degrees from north)
└── allstats_wind_<component>_<country>.csv   output of this notebook, read by 4.1b
```

### Input formats

- **Metadata**: one row per station, with columns for the station identifier, name, latitude, longitude and altitude. Set their names in `METADATA_COLUMNS`.
- **Time series**: a `Date` column, then one column per station named `<station id>_<station name>`, for example `FQPB_Pemba`. The part before the first `_` must match the identifier in the metadata.

### Component selection

Set `COMPONENT` to `"speed"` or `"direction"` and run the notebook once for each.


In [ ]:
# =========================
# COUNTRY AND MAIN FOLDER
# =========================

# Country name used in folder and file names.
COUNTRY = "mozambique"

# Folder with the station files; the output CSV is written here as well (read by 4.1b).
PROJECT_DIR = Path("data/stations") / COUNTRY

# =========================
# INPUT FILES
# =========================

METADATA_FILE = PROJECT_DIR / "stations_metadata.xlsx"
WIND_SPEED_FILE = PROJECT_DIR / "wind_speed_stations.csv"
WIND_DIRECTION_FILE = PROJECT_DIR / "wind_direction_stations.csv"

# Text encoding of the CSV files (e.g. "utf-8" or "latin1").
CSV_ENCODING = "utf-8"

# Column names of the metadata file: notebook name -> name in your file.
METADATA_COLUMNS = {
    "id": "station_id",
    "name": "name",
    "latitude": "latitude",
    "longitude": "longitude",
    "altitude": "altitude",
}

# Optional shapefile used only for the map.
# If you do not have a shapefile, set it to None.
SHAPEFILE = Path("data/shapefiles") / COUNTRY / "moz_admin1.shp"

# =========================
# OUTPUT FOLDER
# =========================

OUTPUT_DIR = PROJECT_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# =========================
# ANALYSIS OPTIONS
# =========================

# Choose "speed" or "direction".
COMPONENT = "speed"

# Bounding box used to select the target area (same as the downscaling notebooks).
# Format required by this notebook: [lat_max, lon_min, lat_min, lon_max]
AREA = [-10.0, 30.0, -27.5, 41.5]

# Delete stations with more than this fraction of missing values.
# 0.95 means that stations with more than 95 percent missing data are removed.
MAX_NAN_FRACTION = 0.95

# Remove statistical outliers with the IQR method.
REMOVE_OUTLIERS = True


## Step 3. Functions

Run this cell without editing it.

The functions below:

1. check that the input parameters are valid;
2. read the metadata and wind data;
3. optionally read the shapefile;
4. clean the station time series;
5. calculate the monthly climatology;
6. merge the climatology with station coordinates and save the final output.


In [ ]:
def validate_inputs(
    component,
    metadata_file,
    wind_speed_file,
    wind_direction_file,
    output_dir,
    area,
):
    """Validate the main user inputs before running the notebook."""
    component = component.lower().strip()

    if component not in {"speed", "direction"}:
        raise ValueError('COMPONENT must be either "speed" or "direction".')

    if not metadata_file.exists():
        raise FileNotFoundError(f"Metadata file not found: {metadata_file}")

    data_file = wind_speed_file if component == "speed" else wind_direction_file
    if not data_file.exists():
        raise FileNotFoundError(f"Wind data file not found: {data_file}")

    if len(area) != 4:
        raise ValueError("AREA must contain four values: [lat_max, lon_min, lat_min, lon_max].")

    output_dir.mkdir(parents=True, exist_ok=True)
    return component, data_file


def read_metadata(metadata_file, columns):
    """Read station metadata, check the required columns and rename them to the notebook names."""
    metadata = pd.read_excel(metadata_file)

    missing_columns = set(columns.values()).difference(metadata.columns)
    if missing_columns:
        raise ValueError(
            "The metadata file is missing these required columns: "
            + ", ".join(sorted(missing_columns))
            + ". Update METADATA_COLUMNS in Step 2."
        )

    metadata = metadata.rename(columns={file_name: name for name, file_name in columns.items()})
    metadata["id"] = metadata["id"].astype(str).str.strip()
    return metadata


def read_wind_data(data_file, encoding="utf-8"):
    """Read wind observations from CSV."""
    return pd.read_csv(data_file, encoding=encoding)


def read_shapefile_if_available(shapefile):
    """Read the optional shapefile if it exists."""
    if shapefile is None:
        print("No shapefile selected. The map will use only station points.")
        return None

    shapefile = Path(shapefile)
    if not shapefile.exists():
        print(f"Shapefile not found: {shapefile}")
        print("The workflow can continue without the shapefile.")
        return None

    import geopandas as gpd

    geometries = gpd.read_file(shapefile)
    return geometries.to_crs("EPSG:4326")


def filter_metadata_by_area(metadata, area):
    """Filter stations using a bounding box."""
    lat_max, lon_min, lat_min, lon_max = area

    return metadata[
        (metadata["longitude"] >= lon_min)
        & (metadata["longitude"] <= lon_max)
        & (metadata["latitude"] >= lat_min)
        & (metadata["latitude"] <= lat_max)
    ].copy()


def plot_stations(metadata, area=None, geometries=None):
    """Plot stations, optionally with administrative boundaries."""
    fig, ax = plt.subplots(figsize=(8, 6))

    if geometries is not None:
        geometries.plot(ax=ax, edgecolor="black", facecolor="none", linewidth=0.8)

    ax.scatter(metadata["longitude"], metadata["latitude"], s=20)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_title("Wind stations")

    if area is not None:
        lat_max, lon_min, lat_min, lon_max = area
        ax.set_xlim(lon_min, lon_max)
        ax.set_ylim(lat_min, lat_max)

    plt.show()


def remove_outliers_iqr(df):
    """Replace outliers with NaN using the interquartile range method."""
    df_clean = df.copy()

    for column in df_clean.columns:
        q1 = df_clean[column].quantile(0.25)
        q3 = df_clean[column].quantile(0.75)
        iqr = q3 - q1

        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr

        df_clean[column] = df_clean[column].where(
            (df_clean[column] >= lower) & (df_clean[column] <= upper)
        )

    return df_clean


def clean_wind_data(df, max_nan_fraction=0.95, remove_outliers=True):
    """Prepare wind data for climatology calculation."""
    df = df.copy()

    if "Date" not in df.columns:
        raise ValueError('The wind CSV must contain a "Date" column.')

    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df = df.dropna(subset=["Date"])
    df = df.set_index("Date").sort_index()
    df = df.apply(pd.to_numeric, errors="coerce")

    nan_fraction = df.isna().mean()
    columns_to_drop = nan_fraction[nan_fraction > max_nan_fraction].index.tolist()
    df = df.loc[:, nan_fraction <= max_nan_fraction]

    print(f"Deleted columns because of missing data: {len(columns_to_drop)}")
    if columns_to_drop:
        print(columns_to_drop)

    if remove_outliers:
        df = remove_outliers_iqr(df)

    return df


def calculate_monthly_climatology(df_clean):
    """Calculate the monthly climatology from daily and monthly means."""
    df_daily = df_clean.resample("D").mean()
    df_monthly = df_daily.resample("ME").mean()

    return df_monthly.groupby(df_monthly.index.month).mean()


def merge_climatology_with_metadata(df_clim, metadata, component):
    """Convert climatology to long format and add station metadata."""
    value_column = f"wind_{component}"

    df_wind = df_clim.copy().reset_index()
    df_wind = df_wind.rename(columns={"Date": "month", "index": "month"})

    df_wind_long = df_wind.melt(
        id_vars="month",
        var_name="station_raw",
        value_name=value_column,
    )

    # Station columns are named "<station id>_<station name>": the id is the part before the first "_".
    df_wind_long["id"] = df_wind_long["station_raw"].astype(str).str.split("_", n=1).str[0].str.strip()

    df_final = df_wind_long.merge(
        metadata[["id", "name", "latitude", "longitude", "altitude"]],
        on="id",
        how="left",
    )

    unmatched = df_final.loc[df_final["latitude"].isna(), "station_raw"].unique()
    if len(unmatched):
        print(f"Stations without metadata (dropped): {list(unmatched)}")
    df_final = df_final.dropna(subset=["latitude", "longitude"])

    return df_final[["month", "id", "name", "latitude", "longitude", "altitude", value_column]]


def save_output(df_final, output_dir, component, country):
    """Save the final table to CSV."""
    output_file = output_dir / f"allstats_wind_{component}_{country}.csv"
    df_final.to_csv(output_file, index=False)
    print(f"Saved output file: {output_file}")
    return output_file


## Step 4. Read inputs

Run this cell after checking the parameters in Step 2.

The cell loads station metadata, the selected wind data file and, if available, the shapefile.


In [ ]:
COMPONENT, WIND_DATA_FILE = validate_inputs(
    component=COMPONENT,
    metadata_file=METADATA_FILE,
    wind_speed_file=WIND_SPEED_FILE,
    wind_direction_file=WIND_DIRECTION_FILE,
    output_dir=OUTPUT_DIR,
    area=AREA,
)

metadata = read_metadata(METADATA_FILE, METADATA_COLUMNS)
wind_data = read_wind_data(WIND_DATA_FILE, CSV_ENCODING)
geometries = read_shapefile_if_available(SHAPEFILE)

print(f"Selected component: {COMPONENT}")
print(f"Metadata rows: {len(metadata)}")
print(f"Wind data shape: {wind_data.shape}")
metadata.head()


## Step 5. Check the station area

This step filters the metadata using the selected bounding box and plots the stations.

Use this step to check that `AREA` is correct before continuing.


In [ ]:
metadata_filtered = filter_metadata_by_area(metadata, AREA)

print(f"Stations inside selected area: {len(metadata_filtered)}")
plot_stations(metadata_filtered, area=AREA, geometries=geometries)
metadata_filtered.head()


## Step 6. Clean the wind data

This step:

1. converts the `Date` column to a datetime index;
2. converts station columns to numeric values;
3. removes stations with too many missing values;
4. optionally removes outliers using the IQR method.


In [ ]:
wind_clean = clean_wind_data(
    wind_data,
    max_nan_fraction=MAX_NAN_FRACTION,
    remove_outliers=REMOVE_OUTLIERS,
)

print(f"Clean data shape: {wind_clean.shape}")
wind_clean.head()


## Step 7. Calculate monthly climatology

This step calculates a monthly climatology for each station.

The output has one row for each month from 1 to 12.


In [ ]:
wind_climatology = calculate_monthly_climatology(wind_clean)

print(f"Monthly climatology shape: {wind_climatology.shape}")
wind_climatology.head(12)


## Step 8. Merge climatology with station metadata

This step converts the station table from wide to long format and adds station name, latitude, longitude and altitude.


In [ ]:
wind_final = merge_climatology_with_metadata(
    df_clim=wind_climatology,
    metadata=metadata,
    component=COMPONENT,
)

# Keep only stations inside the selected area.
lat_max, lon_min, lat_min, lon_max = AREA
wind_final_area = wind_final[
    (wind_final["longitude"] >= lon_min)
    & (wind_final["longitude"] <= lon_max)
    & (wind_final["latitude"] >= lat_min)
    & (wind_final["latitude"] <= lat_max)
].copy()

print(f"Final rows before area filter: {len(wind_final)}")
print(f"Final rows after area filter: {len(wind_final_area)}")
wind_final_area.head()


## Step 9. Save the output CSV

Run this cell to save the final table.

The output file is saved in:

```text
PROJECT_DIR/  (data/stations/<country>/)
```


In [ ]:
output_file = save_output(
    df_final=wind_final_area,
    output_dir=OUTPUT_DIR,
    component=COMPONENT,
    country=COUNTRY,
)

output_file


## Notes for adapting this notebook

To reuse this notebook for another country or another station dataset:

1. change `COUNTRY` (the folder `data/stations/<country>/` follows from it);
2. update the input file names if needed;
3. set `METADATA_COLUMNS` to the column names of your metadata file;
4. name the station columns of the time series `<station id>_<station name>`;
5. update `AREA` using the format `[lat_max, lon_min, lat_min, lon_max]`;
6. run all cells from top to bottom, once for `"speed"` and once for `"direction"`.

The notebook does not modify the original input files. It only creates a new CSV, which 4.1b reads.
